<a href="https://colab.research.google.com/github/Kubaldo01/Analiza_xG/blob/main/analiza_pog_pol.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import numpy as np
import pandas as pd

df = pd.read_csv('/content/Polonia Bytom_Pogo  Grodzisk Mazowiecki_4068759.csv')
team1_name = "Pogoń Grodzisk Mazowiecki"
team2_name = "Polonia Bytom"

df = df.drop_duplicates(subset="id")

df["pogon_goal"] = (
    (df["team_name"] == team1_name) &
    (df["event_type_name"] == "Shot") &
    (df["outcome_name"] == "Goal")
).astype(int)

df["polonia_goal"] = (
    (df["team_name"] == team2_name) &
    (df["event_type_name"] == "Shot") &
    (df["outcome_name"] == "Goal")
).astype(int)

df["goals_pogon"] = df["pogon_goal"].cumsum() - df["pogon_goal"]
df["goals_polonia"] = df["polonia_goal"].cumsum() - df["polonia_goal"]

is_team1_event = df["team_name"] == team1_name

df["score_diff"] = np.where(is_team1_event,
    df["goals_pogon"] - df["goals_polonia"],
    df["goals_polonia"] - df["goals_pogon"]
)

df["match_state"] = np.select(
    [
        df["score_diff"] > 0,
        df["score_diff"] == 0,
        df["score_diff"] < 0
    ],
    [
        "winning",
        "draw",
        "losing"
    ],
    default="unknown"
)

shots = df[df["event_type_name"] == "Shot"].copy()

states = ["winning", "draw", "losing"]

xg = (
    shots.pivot_table(
        index="team_name",
        columns="match_state",
        values="statsbomb_xg",
        aggfunc="sum",
        fill_value=0
    ).round(3)
    .reindex(columns=states, fill_value=0)
    .T
)

xg

team_name,Pogoń Grodzisk Mazowiecki,Polonia Bytom
match_state,,
winning,0.117,0.066
draw,0.609,0.923
losing,0.017,0.079
